In [5]:
import os
os.environ["PYSPARK_PYTHON"]='PYTHON'

In [6]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("BusData").getOrCreate()

In [7]:
# The Raw files in the folder are read
raw_path = "Data/Raw"
files = os.listdir(raw_path)
print(files)

['.ipynb_checkpoints', 'agency.txt', 'calendar.txt', 'calendar_dates.txt', 'distrubtion.csv', 'feed_info.txt', 'routes.txt', 'shapes.txt', 'siri.xml', 'sirisx.xml', 'stops.txt', 'stop_times.txt', 'trips.txt', 'vehicle.csv']


# Read every GTFS txt file from the raw Data

In [8]:
gtfs = {}
for file in files:
    if file.endswith(".txt"):
        name = file.replace(".txt","")
        df = spark.read.csv(os.path.join(raw_path,file),header=True,inferSchema=True)
        gtfs[name] = df
print(gtfs.keys())

dict_keys(['agency', 'calendar', 'calendar_dates', 'feed_info', 'routes', 'shapes', 'stops', 'stop_times', 'trips'])


# Exploring the Data

In [9]:
# Explore each dataset with rows , column and Schema
for name, df in gtfs.items():
    print(name.upper())
    print("-"*70)
    print("Rows :", df.count())
    print("Columns :", len(df.columns))
    df.printSchema()
    df.show(5, truncate=False)

AGENCY
----------------------------------------------------------------------
Rows : 70
Columns : 7
root
 |-- agency_id: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- agency_url: string (nullable = true)
 |-- agency_timezone: string (nullable = true)
 |-- agency_lang: string (nullable = true)
 |-- agency_phone: string (nullable = true)
 |-- agency_noc: string (nullable = true)

+---------+--------------+--------------------------+---------------+-----------+------------+----------+
|agency_id|agency_name   |agency_url                |agency_timezone|agency_lang|agency_phone|agency_noc|
+---------+--------------+--------------------------+---------------+-----------+------------+----------+
|OP10     |M & H Coaches |https://www.traveline.info|Europe/London  |EN         |NULL        |MHCO      |
|OP1092   |Mersey Ferries|https://www.traveline.info|Europe/London  |EN         |NULL        |FRRY      |
|OP10924  |Bee Network   |https://www.traveline.info|Europe/Lo

In [10]:
# Statistical Summary
for name, df in gtfs.items():
    print(f"\n{name}")
    df.describe().show()


agency
+-------+---------+--------------------+--------------------+---------------+-----------+-------------+----------+
|summary|agency_id|         agency_name|          agency_url|agency_timezone|agency_lang| agency_phone|agency_noc|
+-------+---------+--------------------+--------------------+---------------+-----------+-------------+----------+
|  count|       70|                  70|                  70|             70|         70|            2|        70|
|   mean|     NULL|                NULL|                NULL|           NULL|       NULL|         NULL|      NULL|
| stddev|     NULL|                NULL|                NULL|           NULL|       NULL|         NULL|      NULL|
|    min|     OP10|       A and J Taxis|https://www.trave...|  Europe/London|         EN| 01576 203874|      A2BC|
|    max|    OP853|Wright Brothers C...|https://www.trave...|  Europe/London|         EN|0161 205 2000|      WGTB|
+-------+---------+--------------------+--------------------+-----------

In [11]:
# Missing Values
from pyspark.sql.functions import col,isnan,when,count

for name,df in gtfs.items():

    print(f"\n{name}")

    df.select([
        count(when(col(c).isNull(),c)).alias(c)
        for c in df.columns
    ]).show()


agency
+---------+-----------+----------+---------------+-----------+------------+----------+
|agency_id|agency_name|agency_url|agency_timezone|agency_lang|agency_phone|agency_noc|
+---------+-----------+----------+---------------+-----------+------------+----------+
|        0|          0|         0|              0|          0|          68|         0|
+---------+-----------+----------+---------------+-----------+------------+----------+


calendar
+----------+------+-------+---------+--------+------+--------+------+----------+--------+
|service_id|monday|tuesday|wednesday|thursday|friday|saturday|sunday|start_date|end_date|
+----------+------+-------+---------+--------+------+--------+------+----------+--------+
|         0|     0|      0|        0|       0|     0|       0|     0|         0|       0|
+----------+------+-------+---------+--------+------+--------+------+----------+--------+


calendar_dates
+----------+----+--------------+
|service_id|date|exception_type|
+----------+-

In [12]:
# Dublicate Values
for name,df in gtfs.items():
    print(name)
    print("Total :",df.count())
    print("Distinct :",df.distinct().count(),"\n")

agency
Total : 70
Distinct : 70 

calendar
Total : 277
Distinct : 277 

calendar_dates
Total : 24097
Distinct : 24097 

feed_info
Total : 1
Distinct : 1 

routes
Total : 1669
Distinct : 1669 

shapes
Total : 2921218
Distinct : 2921218 

stops
Total : 36607
Distinct : 36607 

stop_times
Total : 5195977
Distinct : 5195977 

trips
Total : 120474
Distinct : 120474 



In [13]:
summary = []

for name, df in gtfs.items():

    summary.append((
        name,
        df.count(),
        len(df.columns),
        df.count() - df.distinct().count()
    ))

summary_df = spark.createDataFrame(
    summary,
    ["Dataset","Rows","Columns","Duplicate Rows"]
)

summary_df.show(truncate=False)

+--------------+-------+-------+--------------+
|Dataset       |Rows   |Columns|Duplicate Rows|
+--------------+-------+-------+--------------+
|agency        |70     |7      |0             |
|calendar      |277    |10     |0             |
|calendar_dates|24097  |3      |0             |
|feed_info     |1      |6      |0             |
|routes        |1669   |5      |0             |
|shapes        |2921218|5      |0             |
|stops         |36607  |9      |0             |
|stop_times    |5195977|10     |0             |
|trips         |120474 |9      |0             |
+--------------+-------+-------+--------------+



# Parsing the xml files

In [14]:
import xml.etree.ElementTree as ET
import pandas as pd

tree = ET.parse("Data/Raw/siri.xml")
root = tree.getroot()

ns = {"s": "http://www.siri.org.uk/siri"}

data = []

for v in root.findall(".//s:VehicleActivity", ns):

    j = v.find("s:MonitoredVehicleJourney", ns)
    loc = j.find("s:VehicleLocation", ns)

    data.append([
        v.findtext("s:RecordedAtTime", namespaces=ns),
        j.findtext("s:LineRef", namespaces=ns),
        j.findtext("s:OperatorRef", namespaces=ns),
        j.findtext("s:VehicleRef", namespaces=ns),
        loc.findtext("s:Latitude", namespaces=ns),
        loc.findtext("s:Longitude", namespaces=ns)
    ])

df_siri = pd.DataFrame(data, columns=[
    "RecordedTime",
    "LineRef",
    "Operator",
    "VehicleRef",
    "Latitude",
    "Longitude"
])

df_siri.head()

,RecordedTime,LineRef,Operator,VehicleRef,Latitude,Longitude
0,2026-07-02T15:10:13+00:00,96A,A2BR,3303,51.981412,-0.233722
1,2026-07-02T17:33:39+00:00,16A,A2BR,3311,52.073968,0.008972
2,2026-07-02T18:12:15+00:00,811,A2BV,A2BV-24,53.368135,-3.069226
3,2026-07-02T19:16:10+00:00,1A,A2BV,A2BV-RE24_TDZ,53.365285,-3.065045
4,2026-07-02T17:11:17+00:00,106,A2BV,A2BV-YJ10_MHK,53.419548,-3.046595


In [15]:
df_siri.to_csv("Data/Raw/vehicle.csv", index=False)

In [16]:
# Nested structure of xml file for the distrubtion data

In [17]:
import pandas as pd
df_sirisx = pd.read_xml("Data/Raw/sirisx.xml")
df_sirisx.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 4 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   ResponseTimestamp          1 non-null      object
 1   ProducerRef                1 non-null      object
 2   ResponseMessageIdentifier  1 non-null      object
 3   SituationExchangeDelivery  1 non-null      object
dtypes: object(4)
memory usage: 160.0+ bytes


In [18]:
print(df_sirisx.columns.tolist())

['ResponseTimestamp', 'ProducerRef', 'ResponseMessageIdentifier', 'SituationExchangeDelivery']


In [19]:
display(df_sirisx)

,ResponseTimestamp,ProducerRef,ResponseMessageIdentifier,SituationExchangeDelivery
0,2026-07-03T02:43:42.835Z,DepartmentForTransport,20e2423c-58d1-457a-ab3d-0c3ae171952f,\n


In [20]:
print(df_sirisx.loc[0, "SituationExchangeDelivery"])

In [21]:
import xml.etree.ElementTree as ET

tree = ET.parse("Data/Raw/sirisx.xml")
root = tree.getroot()

for i, elem in enumerate(root.iter()):
    print(elem.tag)
    if i >= 30:
        break

{http://www.siri.org.uk/siri}Siri
{http://www.siri.org.uk/siri}ServiceDelivery
{http://www.siri.org.uk/siri}ResponseTimestamp
{http://www.siri.org.uk/siri}ProducerRef
{http://www.siri.org.uk/siri}ResponseMessageIdentifier
{http://www.siri.org.uk/siri}SituationExchangeDelivery
{http://www.siri.org.uk/siri}ResponseTimestamp
{http://www.siri.org.uk/siri}Situations
{http://www.siri.org.uk/siri}PtSituationElement
{http://www.siri.org.uk/siri}CreationTime
{http://www.siri.org.uk/siri}ParticipantRef
{http://www.siri.org.uk/siri}SituationNumber
{http://www.siri.org.uk/siri}Version
{http://www.siri.org.uk/siri}Source
{http://www.siri.org.uk/siri}SourceType
{http://www.siri.org.uk/siri}TimeOfCommunication
{http://www.siri.org.uk/siri}VersionedAtTime
{http://www.siri.org.uk/siri}Progress
{http://www.siri.org.uk/siri}ValidityPeriod
{http://www.siri.org.uk/siri}StartTime
{http://www.siri.org.uk/siri}PublicationWindow
{http://www.siri.org.uk/siri}StartTime
{http://www.siri.org.uk/siri}MiscellaneousR

In [22]:
df_sirisx = pd.read_xml(
    "Data/Raw/sirisx.xml",
    xpath=".//ns:PtSituationElement",
    namespaces={"ns": "http://www.siri.org.uk/siri"}
)

df_sirisx.head()

,CreationTime,ParticipantRef,SituationNumber,Version,Source,VersionedAtTime,Progress,ValidityPeriod,PublicationWindow,MiscellaneousReason,Planned,Summary,Description,Consequences,InfoLinks,EquipmentReason
0,2024-08-30T09:07:04.813Z,WestofEngland,18224249-29cf-4f65-a023-11067e7c2b6f,5,\n,2025-07-22T07:29:53.362Z,open,\n,\n,roadworks,True,Live Traffic Update: York Road One Way Closure,Long term works - York Road is closed eastboun...,\n,None,None
1,2024-11-01T15:42:15.905Z,WYCA,2b9e1d8f-b0ee-43a7-8ca5-7334d6fc4587,3,\n,2025-02-19T13:24:15.995Z,open,\n,\n,roadworks,True,"Horsforth Vale, Bletchley Avenue, Bletchley Ro...","Bletchley Avenue, Bletchley Road and Low Hall ...",\n,\n,None
2,2025-03-06T09:16:59Z,WestofEngland,ba174836-2698-4f06-8f22-e6c90aca9a7d,1,\n,2025-03-06T09:16:59Z,open,\n,\n,roadworks,False,Tower Road/Station Road,Due to Temporary Lights on the Junction of Tow...,\n,None,None
3,2025-03-28T11:17:43Z,WestofEngland,8e8d2259-4d28-490b-9785-c08b0fd034bd,4,\n,2025-10-15T08:35:49.810Z,open,\n,\n,roadworks,True,"Road Closure: Victoria Street, Bristol","Victoria Street in Bristol City Centre, northb...",\n,None,None
4,2025-03-31T11:39:08Z,WestofEngland,526571b9-9ead-4d5a-a7f0-511c879be31e,1,\n,2025-03-31T11:39:08Z,open,\n,\n,None,True,Stop closure: The Assembly Rooms,This stop will be closed for approximately 20 ...,\n,None,repairWork


In [23]:
df_sirisx.to_csv("Data/Raw/distrubtion.csv", index=False)

In [24]:
def inspect_pandas(name, df):

    print("\n" + "="*90)
    print(f"DATASET : {name}")
    print("="*90)

    print(f"Rows      : {df.shape[0]}")
    print(f"Columns   : {df.shape[1]}")
    print(f"Duplicates: {df.duplicated().sum()}")

    print("\nData Types")
    print(df.dtypes)

    print("\nMissing Values")
    print(df.isnull().sum())

    print("\nSummary Statistics")
    display(df.describe(include="all"))

    print("\nSample Records")
    display(df.head())

inspect_pandas("Vehicle", df_siri)
inspect_pandas("Disruptions", df_sirisx)


DATASET : Vehicle
Rows      : 27502
Columns   : 6
Duplicates: 0

Data Types
RecordedTime    object
LineRef         object
Operator        object
VehicleRef      object
Latitude        object
Longitude       object
dtype: object

Missing Values
RecordedTime     0
LineRef         14
Operator         0
VehicleRef       0
Latitude         0
Longitude        0
dtype: int64

Summary Statistics


,RecordedTime,LineRef,Operator,VehicleRef,Latitude,Longitude
count,27502,27488,27502,27502,27502,27502
unique,18958,2329,365,25622,20989,21230
top,2026-07-03T01:21:09+00:00,1,TFLO,2010,0,0
freq,40,470,7764,6,111,111



Sample Records


,RecordedTime,LineRef,Operator,VehicleRef,Latitude,Longitude
0,2026-07-02T15:10:13+00:00,96A,A2BR,3303,51.981412,-0.233722
1,2026-07-02T17:33:39+00:00,16A,A2BR,3311,52.073968,0.008972
2,2026-07-02T18:12:15+00:00,811,A2BV,A2BV-24,53.368135,-3.069226
3,2026-07-02T19:16:10+00:00,1A,A2BV,A2BV-RE24_TDZ,53.365285,-3.065045
4,2026-07-02T17:11:17+00:00,106,A2BV,A2BV-YJ10_MHK,53.419548,-3.046595



DATASET : Disruptions
Rows      : 374
Columns   : 16
Duplicates: 0

Data Types
CreationTime           object
ParticipantRef         object
SituationNumber        object
Version                 int64
Source                 object
VersionedAtTime        object
Progress               object
ValidityPeriod         object
PublicationWindow      object
MiscellaneousReason    object
Planned                  bool
Summary                object
Description            object
Consequences           object
InfoLinks              object
EquipmentReason        object
dtype: object

Missing Values
CreationTime             0
ParticipantRef           0
SituationNumber          0
Version                  0
Source                   0
VersionedAtTime          0
Progress                 0
ValidityPeriod           0
PublicationWindow        0
MiscellaneousReason     95
Planned                  0
Summary                  0
Description              0
Consequences             0
InfoLinks              214
Equip

,CreationTime,ParticipantRef,SituationNumber,Version,Source,VersionedAtTime,Progress,ValidityPeriod,PublicationWindow,MiscellaneousReason,Planned,Summary,Description,Consequences,InfoLinks,EquipmentReason
count,374,374,374,374.000000,374,374,374,374,374,279,374,374,374,374,160,95
unique,358,9,374,NaN,1,374,1,1,1,9,2,357,371,1,1,6
top,2025-06-24T09:39:50Z,TfGM,54c6f77a-cfbd-4d80-a516-9346bbd63714,NaN,\n,2026-06-26T07:17:40Z,open,\n,\n,roadworks,True,Victoria - Tram Improvement Works,Due to an emergency road closure on Alan Turin...,\n,\n,maintenanceWork
freq,5,117,1,NaN,374,1,374,374,374,197,298,4,2,374,160,73
mean,NaN,NaN,NaN,3.695187,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,10.370257,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,2.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Sample Records


,CreationTime,ParticipantRef,SituationNumber,Version,Source,VersionedAtTime,Progress,ValidityPeriod,PublicationWindow,MiscellaneousReason,Planned,Summary,Description,Consequences,InfoLinks,EquipmentReason
0,2024-08-30T09:07:04.813Z,WestofEngland,18224249-29cf-4f65-a023-11067e7c2b6f,5,\n,2025-07-22T07:29:53.362Z,open,\n,\n,roadworks,True,Live Traffic Update: York Road One Way Closure,Long term works - York Road is closed eastboun...,\n,None,None
1,2024-11-01T15:42:15.905Z,WYCA,2b9e1d8f-b0ee-43a7-8ca5-7334d6fc4587,3,\n,2025-02-19T13:24:15.995Z,open,\n,\n,roadworks,True,"Horsforth Vale, Bletchley Avenue, Bletchley Ro...","Bletchley Avenue, Bletchley Road and Low Hall ...",\n,\n,None
2,2025-03-06T09:16:59Z,WestofEngland,ba174836-2698-4f06-8f22-e6c90aca9a7d,1,\n,2025-03-06T09:16:59Z,open,\n,\n,roadworks,False,Tower Road/Station Road,Due to Temporary Lights on the Junction of Tow...,\n,None,None
3,2025-03-28T11:17:43Z,WestofEngland,8e8d2259-4d28-490b-9785-c08b0fd034bd,4,\n,2025-10-15T08:35:49.810Z,open,\n,\n,roadworks,True,"Road Closure: Victoria Street, Bristol","Victoria Street in Bristol City Centre, northb...",\n,None,None
4,2025-03-31T11:39:08Z,WestofEngland,526571b9-9ead-4d5a-a7f0-511c879be31e,1,\n,2025-03-31T11:39:08Z,open,\n,\n,None,True,Stop closure: The Assembly Rooms,This stop will be closed for approximately 20 ...,\n,None,repairWork


# Data Understanding and Relationship

In [25]:
gtfs["routes"].select("route_id").show(5)

gtfs["trips"].select("trip_id","route_id","service_id").show(5)

gtfs["stop_times"].select("trip_id","stop_id","arrival_time","departure_time").show(5)

gtfs["stops"].select("stop_id","stop_name").show(5)

gtfs["calendar"].select("service_id").show(5)

df_siri.head()

df_sirisx.head()

+--------+
|route_id|
+--------+
|       8|
|      30|
|      46|
|      77|
|      92|
+--------+
only showing top 5 rows

+--------------------+--------+----------+
|             trip_id|route_id|service_id|
+--------------------+--------+----------+
|VJ008f72c7264e51c...|       8|        60|
|VJ00b3f1406d79a81...|       8|        60|
|VJ04981b88d9836bc...|       8|        60|
|VJ091fbab84e764cc...|       8|        97|
|VJ1017064c449c957...|       8|        60|
+--------------------+--------+----------+
only showing top 5 rows

+--------------------+------------+------------+--------------+
|             trip_id|     stop_id|arrival_time|departure_time|
+--------------------+------------+------------+--------------+
|VJ00022cf38fdf14a...|9400ZZMAFIR2|    06:59:00|      06:59:00|
|VJ00022cf38fdf14a...|9400ZZMACHO2|    07:01:00|      07:01:00|
|VJ00022cf38fdf14a...|9400ZZMASTW2|    07:02:00|      07:02:00|
|VJ00022cf38fdf14a...|9400ZZMAWIT2|    07:05:00|      07:05:00|
|VJ00022cf38fdf1

,CreationTime,ParticipantRef,SituationNumber,Version,Source,VersionedAtTime,Progress,ValidityPeriod,PublicationWindow,MiscellaneousReason,Planned,Summary,Description,Consequences,InfoLinks,EquipmentReason
0,2024-08-30T09:07:04.813Z,WestofEngland,18224249-29cf-4f65-a023-11067e7c2b6f,5,\n,2025-07-22T07:29:53.362Z,open,\n,\n,roadworks,True,Live Traffic Update: York Road One Way Closure,Long term works - York Road is closed eastboun...,\n,None,None
1,2024-11-01T15:42:15.905Z,WYCA,2b9e1d8f-b0ee-43a7-8ca5-7334d6fc4587,3,\n,2025-02-19T13:24:15.995Z,open,\n,\n,roadworks,True,"Horsforth Vale, Bletchley Avenue, Bletchley Ro...","Bletchley Avenue, Bletchley Road and Low Hall ...",\n,\n,None
2,2025-03-06T09:16:59Z,WestofEngland,ba174836-2698-4f06-8f22-e6c90aca9a7d,1,\n,2025-03-06T09:16:59Z,open,\n,\n,roadworks,False,Tower Road/Station Road,Due to Temporary Lights on the Junction of Tow...,\n,None,None
3,2025-03-28T11:17:43Z,WestofEngland,8e8d2259-4d28-490b-9785-c08b0fd034bd,4,\n,2025-10-15T08:35:49.810Z,open,\n,\n,roadworks,True,"Road Closure: Victoria Street, Bristol","Victoria Street in Bristol City Centre, northb...",\n,None,None
4,2025-03-31T11:39:08Z,WestofEngland,526571b9-9ead-4d5a-a7f0-511c879be31e,1,\n,2025-03-31T11:39:08Z,open,\n,\n,None,True,Stop closure: The Assembly Rooms,This stop will be closed for approximately 20 ...,\n,None,repairWork


In [26]:
gtfs["routes"].select("route_short_name").show(20, False)

+----------------+
|route_short_name|
+----------------+
|4               |
|4S              |
|14              |
|10              |
|11              |
|220             |
|221             |
|643             |
|316             |
|317             |
|127A            |
|288             |
|576             |
|256             |
|314             |
|325             |
|330             |
|346             |
|368             |
|374             |
+----------------+
only showing top 20 rows



In [27]:
df_siri["LineRef"].head(20)

0     96A
1     16A
2     811
3      1A
4     106
5      73
6      81
7      1C
8      91
9      1C
10     82
11     R2
12     R3
13     34
14     14
15     2C
16     12
17     13
18    58L
19     13
Name: LineRef, dtype: object

In [28]:
from pyspark.sql.functions import col, upper, trim

vehicle_spark = spark.createDataFrame(df_siri)
vehicle_with_agency = vehicle_spark.join(
    gtfs["agency"],
    upper(trim(vehicle_spark.Operator)) == upper(trim(gtfs["agency"].agency_noc)),
    "left"
)
matched = vehicle_with_agency.join(
    gtfs["routes"],
    (vehicle_with_agency.agency_id == gtfs["routes"].agency_id) &
    (upper(trim(vehicle_with_agency.LineRef)) == upper(trim(gtfs["routes"].route_short_name))),
    "inner"
)

In [29]:
print(
    round(
        matched.count() / vehicle_spark.count() * 100,
        2
    ),
    "% matched"
)

13.7 % matched


In [34]:
total = vehicle_spark.count()

# How many vehicles found a matching agency at all?
agency_matched = vehicle_with_agency.filter(col("agency_id").isNotNull()).count()

# Of those, how many also matched a route?
route_matched = matched.count()

print("Total vehicle records:", total)
print("Matched an agency:", agency_matched, f"({round(agency_matched/total*100,1)}%)")
print("Matched agency AND route:", route_matched, f"({round(route_matched/total*100,1)}%)")

Total vehicle records: 27502
Matched an agency: 5379 (19.6%)
Matched agency AND route: 3768 (13.7%)


In [36]:
vehicle_ops = set(row["Operator"] for row in vehicle_spark.select("Operator").distinct().collect())
agency_nocs = set(row["agency_noc"] for row in gtfs["agency"].select("agency_noc").distinct().collect())

overlap = vehicle_ops.intersection(agency_nocs)
print("Distinct operator codes in vehicle data:", len(vehicle_ops))
print("Distinct agency_noc codes in agency table:", len(agency_nocs))
print("Codes that appear in BOTH:", len(overlap))

Distinct operator codes in vehicle data: 365
Distinct agency_noc codes in agency table: 70
Codes that appear in BOTH: 57


In [30]:
gtfs["routes"] \
    .groupBy("route_short_name") \
    .agg(count("*").alias("count")) \
    .filter("count > 1") \
    .orderBy("count", ascending=False) \
    .show(20, False)

+----------------+-----+
|route_short_name|count|
+----------------+-----+
|1               |13   |
|2               |13   |
|7               |9    |
|8               |9    |
|6               |9    |
|3               |8    |
|10              |8    |
|14              |8    |
|15              |7    |
|42              |7    |
|4               |7    |
|52              |6    |
|5               |6    |
|17              |6    |
|60              |6    |
|41              |6    |
|82              |6    |
|9               |6    |
|781             |6    |
|685             |5    |
+----------------+-----+
only showing top 20 rows



In [31]:
gtfs["routes"] \
    .filter("route_short_name = '14'") \
    .show(truncate=False)

+--------+---------+----------------+---------------+----------+
|route_id|agency_id|route_short_name|route_long_name|route_type|
+--------+---------+----------------+---------------+----------+
|46      |OP6      |14              |NULL           |3         |
|4449    |OP132    |14              |NULL           |3         |
|4450    |OP302    |14              |NULL           |3         |
|4456    |OP255    |14              |NULL           |3         |
|43906   |OP258    |14              |NULL           |3         |
|78779   |OP326    |14              |NULL           |3         |
|132122  |OP260    |14              |NULL           |3         |
|132158  |OP260    |14              |NULL           |3         |
+--------+---------+----------------+---------------+----------+



In [32]:
gtfs["agency"].show(truncate=False)

+---------+--------------------------------+--------------------------+---------------+-----------+------------+----------+
|agency_id|agency_name                     |agency_url                |agency_timezone|agency_lang|agency_phone|agency_noc|
+---------+--------------------------------+--------------------------+---------------+-----------+------------+----------+
|OP10     |M & H Coaches                   |https://www.traveline.info|Europe/London  |EN         |NULL        |MHCO      |
|OP1092   |Mersey Ferries                  |https://www.traveline.info|Europe/London  |EN         |NULL        |FRRY      |
|OP10924  |Bee Network                     |https://www.traveline.info|Europe/London  |EN         |NULL        |BNSM      |
|OP11023  |Bee Network                     |https://www.traveline.info|Europe/London  |EN         |NULL        |BNDB      |
|OP11056  |Bee Network                     |https://www.traveline.info|Europe/London  |EN         |NULL        |BNVB      |
|OP11122

In [33]:
gtfs["routes"].show(10, truncate=False)

+--------+---------+----------------+---------------+----------+
|route_id|agency_id|route_short_name|route_long_name|route_type|
+--------+---------+----------------+---------------+----------+
|8       |OP6      |4               |NULL           |3         |
|30      |OP6      |4S              |NULL           |3         |
|46      |OP6      |14              |NULL           |3         |
|77      |OP6      |10              |NULL           |3         |
|92      |OP6      |11              |NULL           |3         |
|753     |OP13829  |220             |NULL           |3         |
|754     |OP13829  |221             |NULL           |3         |
|950     |OP326    |643             |NULL           |3         |
|1511    |OP31     |316             |NULL           |3         |
|1512    |OP31     |317             |NULL           |3         |
+--------+---------+----------------+---------------+----------+
only showing top 10 rows

